In [1]:
!pip3 install lightning
!pip3 install h5py

  Obtaining dependency information for lightning from https://files.pythonhosted.org/packages/08/f0/617fede29ec0684b310bd2a0c880e640c60b9f2d64ed3d77e2bca2c13bf9/lightning-2.1.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.5 MB/s eta 0:00:00


In [2]:
import h5py
import numpy as np
import torch.utils.data as data
import matplotlib.pyplot as plt

import lightning.pytorch as L
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
import pandas as pd
import numpy as np
import os

# Data

In [3]:
class RadarDatasetNew(data.Dataset):

    def __init__(self, list_of_files, in_seq_len=4, out_seq_len=12, mode='overlap', with_time=False):
        self.in_seq_len = in_seq_len
        self.out_seq_len = out_seq_len
        self.seq_len = in_seq_len + out_seq_len
        self.with_time = with_time
        self.__prepare_timestamps_mapping(list_of_files)
        self.__prepare_sequences(mode)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        data = np.zeros((0,252,252))
        inputs,targets = None,None
        for n,timestamp in enumerate(self.sequences[index]):
            if n < self.in_seq_len:
                with h5py.File(self.timestamp_to_file[timestamp]) as d:
                    new_data = np.array(d[timestamp]['intensity']) + 1
                data = np.vstack([data,np.expand_dims(new_data,axis=0)])
            elif n == self.in_seq_len:
                inputs = data.astype('float32')
                with h5py.File(self.timestamp_to_file[timestamp]) as d:
                    data = np.expand_dims(np.array(d[timestamp]['intensity'])+1,axis=0)
            else:
                with h5py.File(self.timestamp_to_file[timestamp]) as d:
                    new_data = np.array(d[timestamp]['intensity']) + 1
                data = np.vstack([data,np.expand_dims(new_data,axis=0)])
        if inputs is None:
            inputs = data.astype('float32')
            targets = inputs[self.in_seq_len:]
        else:
            targets = data.astype('float32')
        
        if self.with_time:
            return (inputs, self.sequences[index][-1]), targets
        else:
            return inputs, targets

    def __prepare_timestamps_mapping(self, list_of_files):
        self.timestamp_to_file = {}
        for filename in list_of_files:
            with h5py.File(filename) as d:
                self.timestamp_to_file = {
                    **self.timestamp_to_file,
                    **dict(map(lambda x: (x, filename), d.keys()))
                }

    def __prepare_sequences(self, mode):
        timestamps = np.unique(sorted(self.timestamp_to_file.keys()))
        if mode == 'sequentially':
            self.sequences = [
                timestamps[index * self.seq_len: (index + 1) * self.seq_len]
                for index in range(len(timestamps) // self.seq_len)
            ]
        elif mode == 'overlap':
            self.sequences = [
                timestamps[index: index + self.seq_len]
                for index in range(len(timestamps) - self.seq_len + 1)
            ]
        else:
            raise Exception(f'Unknown mode {mode}')
        self.sequences = list(filter(
            lambda x: int(x[-1]) - int(x[0]) == (self.seq_len - 1) * 600,
            self.sequences
        ))

In [4]:
def process_test(model, test_loader, output_file='output.hdf5'):
    model.eval()
    with h5py.File(output_file, mode='w') as f_out:
        for index, item in tqdm(enumerate(test_loader)):
            (inputs, last_input_timestamp), _ = item
            output = model(inputs)
            for index in range(output.shape[1]):
                timestamp_out = str(int(last_input_timestamp[-1]) + 600 * (index + 1))
                f_out.create_group(timestamp_out)
                f_out[timestamp_out].create_dataset(
                    'intensity',
                    data=output[0, index].cpu().detach().numpy()-1
                )
        

def prepare_data_loaders_new(train_batch_size=4, valid_batch_size=1, test_batch_size=1):
    train_dataset = RadarDatasetNew([
        '/kaggle/input/ycup2023-weather/prepared_train/2021-01-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-02-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-03-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-04-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-06-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-07-train.hdf5', 
        '/kaggle/input/ycup2023-weather/prepared_train/2021-09-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-10-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-11-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-12-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-05-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-08-train.hdf5',])
    valid_dataset = RadarDatasetNew([
        '/kaggle/input/ycup2023-weather/prepared_train/2021-05-train.hdf5',
        '/kaggle/input/ycup2023-weather/prepared_train/2021-08-train.hdf5',
    ])
    test_dataset = RadarDatasetNew(['/kaggle/input/ycup2023-weather/prepared-test.hdf5'], out_seq_len=0, with_time=True)
    train_loader = data.DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, num_workers=4)
    valid_loader = data.DataLoader(valid_dataset, batch_size=valid_batch_size, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)
    return train_loader, valid_loader, test_loader

In [5]:
class RainNet(nn.Module):

    def __init__(self):
        ksz = 64
        super().__init__()
        self.conv11 = nn.Conv2d(in_channels=4,out_channels=ksz, kernel_size=(3,3),padding=(1,1))
        self.conv12 = nn.Conv2d(in_channels=ksz,out_channels=ksz, kernel_size=(3,3),padding=(1,1))
        
        self.conv21 = nn.Conv2d(in_channels=ksz,out_channels=ksz*2, kernel_size=(3,3),padding=(1,1))
        self.conv22 = nn.Conv2d(in_channels=ksz*2,out_channels=ksz*2, kernel_size=(3,3),padding=(1,1))
        
        self.conv31 = nn.Conv2d(in_channels=ksz*2,out_channels=ksz*4, kernel_size=(3,3),padding=(1,1))
        self.conv32 = nn.Conv2d(in_channels=ksz*4,out_channels=ksz*4, kernel_size=(3,3),padding=(1,1))
        
        self.conv41 = nn.Conv2d(in_channels=ksz*4,out_channels=ksz*8, kernel_size=(3,3),padding=(1,1))
        self.conv42 = nn.Conv2d(in_channels=ksz*8,out_channels=ksz*8, kernel_size=(3,3),padding=(1,1))
        self.drop4 = nn.Dropout(p=0.5)
        
        self.conv51 = nn.Conv2d(in_channels=ksz*8,out_channels=ksz*16, kernel_size=(3,3),padding=(1,1))
        self.conv52 = nn.Conv2d(in_channels=ksz*16,out_channels=ksz*16, kernel_size=(3,3),padding=(1,1))
        self.drop5 = nn.Dropout(p=0.5)
        
        self.up6 = nn.Upsample(size=(31,31))
        self.drop6 = nn.Dropout(p=0.5)
        self.conv61 = nn.Conv2d(in_channels=ksz*24,out_channels=ksz*8, kernel_size=(3,3),padding=(1,1))
        self.conv62 = nn.Conv2d(in_channels=ksz*8,out_channels=ksz*8, kernel_size=(3,3),padding=(1,1))
        
        self.up7 = nn.Upsample(size=(63,63))
        self.drop7 = nn.Dropout(p=0.5)
        self.conv71 = nn.Conv2d(in_channels=ksz*12,out_channels=ksz*4, kernel_size=(3,3),padding=(1,1))
        self.conv72 = nn.Conv2d(in_channels=ksz*4,out_channels=ksz*4, kernel_size=(3,3),padding=(1,1))
        
        self.up8 = nn.Upsample(size=(126,126))
        self.drop8 = nn.Dropout(p=0.5)
        self.conv81 = nn.Conv2d(in_channels=ksz*6,out_channels=ksz*2, kernel_size=(3,3),padding=(1,1))
        self.conv82 = nn.Conv2d(in_channels=ksz*2,out_channels=ksz*2, kernel_size=(3,3),padding=(1,1))
        
        self.up9 = nn.Upsample(size=(252,252))
        self.drop9 = nn.Dropout(p=0.5)
        self.conv91 = nn.Conv2d(in_channels=ksz*3,out_channels=ksz, kernel_size=(3,3),padding=(1,1))
        self.conv92 = nn.Conv2d(in_channels=ksz,out_channels=ksz, kernel_size=(3,3),padding=(1,1))
        
        self.conv10 = nn.Conv2d(in_channels=ksz,out_channels=12, kernel_size=(3,3),padding=(1,1))
        
        self.relu = torch.relu
        self.max_pool = nn.MaxPool2d(2)

    def forward(self, X):
        num_batch, timestamps, height, width = X.size()
        conv1f = self.conv11(X)
        conv1f_relu = self.relu(conv1f)
        conv1s = self.conv12(conv1f_relu)
        conv1s_relu = self.relu(conv1s)
        pool1 = self.max_pool(conv1s_relu)
        
        conv2f = self.conv21(pool1)
        conv2f_relu = self.relu(conv2f)
        conv2s = self.relu(self.conv22(conv2f_relu))
        conv2s_relu = self.relu(conv2s)
        pool2 = self.max_pool(conv2s_relu)

        
        conv3f = self.conv31(pool2)
        conf3f_relu = self.relu(conv3f)
        conv3s = self.conv32(conv3f)
        conv3s_relu = self.relu(conv3s)
        pool3 = self.max_pool(conv3s_relu)
        
        conv4f = self.conv41(pool3)
        conv4f_relu = self.relu(conv4f)
        conv4s = self.conv42(conv4f_relu)
        conv4s_relu = self.relu(conv4s)
        pool4 = self.drop4(self.max_pool(conv4s_relu))
        
        conv5f = self.conv51(pool4)
        conv5f_relu = self.relu(conv5f)
        conv5s = self.conv52(conv5f_relu)
        conv5s_relu = self.relu(conv5s)
        pool5 = self.drop5(conv5s_relu)
        
        up6 = self.up6(pool5)
        up6_cnt = torch.concat([up6,conv4s_relu],axis=1)
        conv6f = self.conv61(self.drop6(up6_cnt))
        conv6f_relu = self.relu(conv6f)
        conv6s = self.conv62(conv6f_relu)
        conv6s_relu = self.relu(conv6s)
        
        up7 = self.up7(conv6s_relu)
        up7_cnt = torch.concat([up7,conv3s_relu],axis=1)
        conv7f = self.conv71(self.drop7(up7_cnt))
        conv7f_relu = self.relu(conv7f)
        conv7s = self.conv72(conv7f_relu)
        conv7s_relu = self.relu(conv7s)
        
        up8 = self.up8(conv7s_relu)
        up8_cnt = torch.concat([up8,conv2s_relu],axis=1)
        conv8f = self.conv81(self.drop8(up8_cnt))
        conv8f_relu = self.relu(conv8f)
        conv8s = self.conv82(conv8f_relu)
        conv8s_relu = self.relu(conv8s)
        
        up9 = self.up9(conv8s_relu)
        up9_cnt = torch.concat([up9,conv1s_relu],axis=1)
        conv9f = self.conv91(self.drop9(up9_cnt))
        conv9f_relu = self.relu(conv9f)
        conv9s = self.conv92(conv9f_relu)
        conv9s_relu = self.relu(conv9s)
        
        conv10 = self.conv10(conv9s_relu)
        
        return conv10
    
    
class RainNetModel(L.LightningModule):

    def __init__(self):
        super().__init__()
        self.model = RainNet()
        self.training_step_outputs = []

    def forward(self, x):
        x = x.to(device=self.model.conv11.weight.device)
        output = self.model(x)
        return output
    
    @staticmethod
    def _augmentante(x):
        return torch.cat([x,torch.rot90(x,dims=[2,3]),torch.rot90(x,dims=[2,3],k=2),torch.rot90(x,dims=[2,3],k=3)])

    def training_step(self, batch):
        x, y = batch
        x,y = self._augmentante(x), self._augmentante(y)
        out = self.forward(x)
        loss = torch.mean(torch.sqrt(torch.sum(F.mse_loss(y,out,reduction='none')*(y > 0.01),axis=(0,2,3))/y.size()[0]))
        self.training_step_outputs.append(loss)
        self.log("train_loss", loss)
        return loss
    
    def on_train_epoch_end(self) -> None:
        loss = torch.mean(torch.stack(self.training_step_outputs))
        self.training_step_outputs.clear()
        print('Train loss ',loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=9e-5,weight_decay=0.001)
        lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.75)
        return [optimizer],[lr_scheduler]

In [6]:
def main(tensorboard_path='/kaggle/working/tensorboard',max_epochs=5):
    L.seed_everything(123321)
    train_loader, valid_loader, test_loader = prepare_data_loaders_new()
    model = RainNetModel()
    checkpoint_callback = L.callbacks.ModelCheckpoint(every_n_epochs=1,save_on_train_epoch_end=True,save_top_k=-1)
    lr_callback = L.callbacks.LearningRateMonitor(logging_interval='epoch')
    trainer = L.Trainer(
            logger=L.loggers.TensorBoardLogger(save_dir=tensorboard_path),
            max_epochs=max_epochs,
            deterministic=True,
            callbacks=[checkpoint_callback, lr_callback]
    )
    trainer.fit(model, train_loader)
    for epoch in range(max_epochs):
        path = f"{tensorboard_path}/lightning_logs/version_0/checkpoints"
        f_name = [_f for _f in os.listdir(path) if _f.find(f'epoch={epoch}') >= 0][0]
        model = RainNetModel.load_from_checkpoint(path + '/' + f_name)
        process_test(model, test_loader, output_file=f"best_model_{epoch+1}_epoch_output.hdf5")

In [1]:
main()

NameError: name 'main' is not defined